# LV6 - CNN klasifikacija na CIFAR-10

Notebook je organizovan po podzadatcima (a-i) iz upute. Pokreni celije redom u Colab-u.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import SGD

np.random.seed(42)

## a) Ucitavanje CIFAR-10 i ispis oblika

In [ ]:
(trainX, trainy), (testX, testy) = cifar10.load_data()

print('trainX shape:', trainX.shape)
print('trainy shape:', trainy.shape)
print('testX shape :', testX.shape)
print('testy shape :', testy.shape)

## b) Prikaz prvih 9 slika + komentar

In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

plt.figure(figsize=(8, 8))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(trainX[i])
    plt.title(class_names[int(trainy[i])])
    plt.axis('off')
plt.tight_layout()
plt.show()

**Komentar (za b):**
- Slike su male (32x32), niske rezolucije u odnosu na moderne fotografije.
- Zbog niske rezolucije gube se fini detalji, sto oteza razdvajanje slicnih klasa (npr. cat vs dog).
- Istovremeno, mali ulaz ubrzava treniranje i smanjuje memorijske zahtjeve.

## c) Priprema podataka (one-hot + normalizacija [0,1])

In [ ]:
trainX = trainX.astype('float32') / 255.0
testX = testX.astype('float32') / 255.0

trainY = to_categorical(trainy, 10)
testY = to_categorical(testy, 10)

print('trainX dtype/min/max:', trainX.dtype, trainX.min(), trainX.max())
print('trainY shape:', trainY.shape)
print('testY shape :', testY.shape)

## d-e) Baseline model + kompiliranje

In [ ]:
def define_baseline_model():
    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)))
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dense(10, activation='softmax'))

    opt = SGD(learning_rate=0.001, momentum=0.9)
    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

baseline_model = define_baseline_model()
baseline_model.summary()

## f) Treniranje i evaluacija baseline modela

In [ ]:
history_baseline = baseline_model.fit(
    trainX, trainY,
    epochs=15,
    batch_size=64,
    validation_data=(testX, testY),
    verbose=1
)

test_loss_base, test_acc_base = baseline_model.evaluate(testX, testY, verbose=0)
print(f'Baseline test accuracy: {test_acc_base:.4f}')
print(f'Baseline test loss    : {test_loss_base:.4f}')

**Komentar (za f):**
- Ocekivana tacnost za ovakav baseline je cesto oko 0.75-0.82 (zavisno od run-a i okoline).
- Rezultat je solidan za pocetni model, ali postoji prostor za poboljsanje regularizacijom i dubljom arhitekturom.

## g) Plot tacnosti i gubitka kroz epohe

In [ ]:
def plot_history(history, title_prefix='Model'):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, label='train acc')
    plt.plot(epochs, val_acc, label='val acc')
    plt.title(f'{title_prefix} - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, label='train loss')
    plt.plot(epochs, val_loss, label='val loss')
    plt.title(f'{title_prefix} - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

plot_history(history_baseline, 'Baseline')

**Komentar (za g):**
- Ako train accuracy nastavlja rasti, a val accuracy stagnira ili pada, to je znak overfitting-a.
- Ako loss i val_loss prate slican trend bez velikog razilaženja, generalizacija je bolja.

## h) Dodavanje Dropout(0.5) prije prvog Dense sloja

In [ ]:
def define_dropout_model_v1():
    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)))
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))

    model.add(Flatten())
    model.add(Dropout(0.5))
    model.add(Dense(128, activation='relu'))
    model.add(Dense(10, activation='softmax'))

    opt = SGD(learning_rate=0.001, momentum=0.9)
    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

dropout_model_v1 = define_dropout_model_v1()
history_dropout_v1 = dropout_model_v1.fit(
    trainX, trainY,
    epochs=15,
    batch_size=64,
    validation_data=(testX, testY),
    verbose=1
)

test_loss_d1, test_acc_d1 = dropout_model_v1.evaluate(testX, testY, verbose=0)
print(f'Dropout v1 test accuracy: {test_acc_d1:.4f}')
print(f'Dropout v1 test loss    : {test_loss_d1:.4f}')

plot_history(history_dropout_v1, 'Dropout v1 (0.5 prije Dense)')

**Komentar (za h):**
- Dropout 0.5 obicno smanji overfitting, ali moze i malo usporiti ucenje.
- Uporedi train/val krive sa baseline modelom i zakljuci da li je generalizacija bolja.

## i) Finalni model (dublji CNN + vise Dropout slojeva), 30 epoha

In [ ]:
def define_final_model():
    model = Sequential()

    model.add(Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)))
    model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))

    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))

    model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))

    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(10, activation='softmax'))

    opt = SGD(learning_rate=0.001, momentum=0.9)
    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

final_model = define_final_model()
final_model.summary()

In [ ]:
history_final = final_model.fit(
    trainX, trainY,
    epochs=30,
    batch_size=64,
    validation_data=(testX, testY),
    verbose=1
)

test_loss_final, test_acc_final = final_model.evaluate(testX, testY, verbose=0)
print(f'Final model test accuracy: {test_acc_final:.4f}')
print(f'Final model test loss    : {test_loss_final:.4f}')

plot_history(history_final, 'Final model')

**Komentar (za i):**
- Ocekivano je da finalni model postigne visu tacnost od baseline modela.
- Overfitting procjenjujes poredjenjem train i validation krivih: sto manji i stabilniji gap, to bolja generalizacija.